In [2]:
import random
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights
import numpy as np
import cv2
import torchvision.models.segmentation
import torch


In [3]:
imageSize=[600,600]
imgPath="./Datasets/LabPics Chemistry/Train/9Train/Image.jpg"

In [4]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')   # train on the GPU or on the CPU, if a GPU is not available
model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)  # load an instance segmentation model pre-trained pre-trained on COCO
in_features = model.roi_heads.box_predictor.cls_score.in_features  # get number of input features for the classifier
model.roi_heads.box_predictor = FastRCNNPredictor(in_features,num_classes=2)  # replace the pre-trained head with a new one
model.load_state_dict(torch.load("1000.torch"))
model.to(device)# move model to the right devic
model.eval()

MaskRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(in

In [5]:
images=cv2.imread(imgPath)

In [6]:
images=cv2.imread(imgPath)
images = cv2.resize(images, imageSize, cv2.INTER_LINEAR)
images = torch.as_tensor(images, dtype=torch.float32).unsqueeze(0)
images=images.swapaxes(1, 3).swapaxes(2, 3)
images = list(image.to(device) for image in images)

In [7]:
with torch.no_grad():
    pred = model(images)

In [8]:
pred

[{'boxes': tensor([[145.4581,  16.9327, 397.4516, 542.4893],
          [ 40.6666, 113.6862, 323.5745, 556.2235],
          [113.1090, 245.0000, 520.5363, 582.2678],
          [212.7799,  26.2725, 490.2067, 470.4171],
          [311.8297,  16.9042, 585.5308, 557.9437],
          [  0.0000,  16.1678, 600.0000, 543.3353],
          [437.4717, 257.0370, 600.0000, 515.2206],
          [ 24.9688,   0.0000, 160.9509, 428.0826],
          [457.5031, 329.1724, 583.9487, 590.4513],
          [459.4274, 215.9523, 585.3393, 347.7341],
          [418.3150, 209.0731, 528.0281, 536.3174],
          [439.3782, 157.1039, 584.3027, 270.8764],
          [453.0286, 115.8870, 594.9263, 404.4381],
          [404.5360,  10.8202, 532.0433, 348.6151],
          [306.0197, 241.4894, 453.2723, 557.8827],
          [511.8703, 415.8229, 595.9324, 599.5447],
          [492.8637,  45.9890, 600.0000, 345.7706],
          [  0.0000, 443.7274, 557.5903, 595.7257],
          [165.0421, 442.3800, 411.6516, 593.8303],
   

In [9]:
im= images[0].swapaxes(0, 2).swapaxes(0, 1).detach().cpu().numpy().astype(np.uint8)
im2 = im.copy()

In [12]:

for i in range(len(pred[0]['masks'])):
    msk=pred[0]['masks'][i,0].detach().cpu().numpy()
    scr=pred[0]['scores'][i].detach().cpu().numpy()
    if scr>0.6 :
        im2[:,:,0][msk>0.5] = random.randint(0,255)
        im2[:, :, 1][msk > 0.5] = random.randint(0,255)
        im2[:, :, 2][msk > 0.5] = random.randint(0, 255)


In [13]:
cv2.imshow(str(scr), np.hstack([im,im2]))
cv2.waitKey() 

27

: 